In [6]:
import xarray as xr
import cfgrib
import numpy as np
from datetime import datetime
import glob

def gfs_ensemble_grib_to_netcdf(input_file, output_file, 
                              bbox=None, variables=None, 
                              levels=None
                              ) :

    
    # Open the GRIB file using cfgrib
    try:
        # For ensemble data, we need to open with the ensemble dimension
        ds = xr.open_dataset(input_file, engine='cfgrib', 
                            backend_kwargs={'filter_by_keys': {'typeOfLevel': 'isobaricInhPa'}})
    except Exception as e:
        print(f"Error opening GRIB file: {e}")
        return

    # Print available variables for reference
    print("Available variables:", list(ds.data_vars.keys()))
    
    # Apply subsetting
    if variables is not None:
        ds = ds[variables]
    if levels is not None and 'isobaricInhPa' in ds.dims:
        ds = ds.sel(isobaricInhPa=levels)

    
    if bbox is not None:
        min_lon, min_lat, max_lon, max_lat = bbox
        # Handle longitude wrapping if needed
        if min_lon < 0:
            min_lon += 360
        if max_lon < 0:
            max_lon += 360
            
        # Select spatial subset
        ds = ds.where(
            (ds.longitude >= min_lon) & 
            (ds.longitude <= max_lon) & 
            (ds.latitude >= min_lat) & 
            (ds.latitude <= max_lat),
            drop=True
        )

    # Add metadata
    ds.attrs['history'] = f"Processed by GFS ensemble converter on {datetime.now()}"
    ds.attrs['source'] = input_file
    
    # Save to NetCDF
    try:
        ds.to_netcdf(output_file)
        print(f"Successfully saved subset to {output_file}")
    except Exception as e:
        print(f"Error saving NetCDF file: {e}")
        

def merge_gfs_ensemble(grib_files, members , lead_times , output_nc, 
                      bbox=None, variables=None, 
                      levels=None):
    """
    Merge multiple GFS ensemble members and lead times into a single NetCDF file.
    
    Parameters:
        grib_files (list): List of GRIB file paths or glob pattern
        output_nc (str): Output NetCDF file path
        bbox (tuple): (min_lon, min_lat, max_lon, max_lat) for spatial subset
        variables (list): Variables to extract (None for all)
        levels (list): Pressure levels to extract (None for all)
        time_dim (str): Time dimension name ('step' or 'time')
    """
    try:
        
        # Open each file and combine into a single dataset
        ds_list = []
        for ii , file in enumerate( grib_files ):
            try:
                # Open with appropriate filters
                backend_kwargs = {
                    'filter_by_keys': {
                        'typeOfLevel': 'isobaricInhPa' if levels else None,
                        'level': levels
                    }
                }
                
                # Open single file
                ds = xr.open_dataset(file, engine='cfgrib', 
                                    backend_kwargs=backend_kwargs)
                
                # Extract member number from filename if not in data
                if 'member' not in ds.dims :
                    ds = ds.expand_dims({'member': [int( members[ii] )]})
                if 'lead_time' not in ds.dims :
                    ds = ds.expand_dims({'lead_time':[int( lead_times[ii] )]})
                
                ds_list.append(ds)
                
            except Exception as e:
                print(f"Error processing {file}: {e}")
                continue
        
        if not ds_list:
            raise ValueError("No valid GRIB files processed")
        
        # Combine all datasets along ensemble and time dimensions
        print("Merging datasets...")
        combined = xr.combine_by_coords(
            ds_list,
            combine_attrs='drop_conflicts'
        )
        
        # Apply spatial subset if requested
        if bbox:
            min_lon, min_lat, max_lon, max_lat = bbox
            # Handle longitude wrapping
            combined = combined.assign_coords({
                'longitude': (combined.longitude + 360) % 360
            }).sortby('longitude')
            
            combined = combined.where(
                (combined.longitude >= min_lon) & 
                (combined.longitude <= max_lon) & 
                (combined.latitude >= min_lat) & 
                (combined.latitude <= max_lat),
                drop=True
            )
        
        # Select variables if specified
        if variables:
            combined = combined[variables]
        
        # Add metadata
        combined.attrs['history'] = f"Merged by GFS processor on {datetime.now()}"
        combined.attrs['source_files'] = grib_files
        
        # Save to NetCDF with compression
        encoding = {var: {'zlib': True, 'complevel': 4} for var in combined.data_vars}
        print(f"Saving to {output_nc}...")
        combined.to_netcdf(output_nc, encoding=encoding)
        print("Successfully saved merged ensemble data")
        
    except Exception as e:
        print(f"Error in merge_gfs_ensemble: {e}")
        raise

        



In [ ]:
# Example usage
ens_size = 30
date = '20231216'
init_time = '00'
data_path = '/home/jruiz/datosdemerzel/GFSDATA/gefs.' + date +'/' + init_time + '/pgrb2b/'
output_file = '/data_path/gfs_ens.nc'
ens_members = np.arange(0,ens_size+1).astype(str)    #.zfill(3)
print(ens_members)

#if __name__ == "__main__":
# Input parameters
levels = [1000.,  900.,  800.,  700.,  600.,  500.,  400.,  300.]
# Define subset parameters
bbox = (-75, -60, -40, -20 )   # Continental US approx
variables = ['t', 'q', 'u', 'v' , 'gh']  # Temperature, humidity, wind components
ensemble_members = None #[0, 1, 2, 3]  # First few ensemble members
time_indices = [0]  # First time step

file_list = []
members_list = []
lead_time_list = [] 
for imem in ens_members :
    print('My member is ',imem)
    str_mem = imem.zfill(3)
    file_list += glob.glob( data_path + str_mem + '/*.pgrb2.*[!idx][!nc]' ) 
        
    members_list = []
    lead_time_list = []
    for my_file in file_list :
        members_list.append( str_mem )
        lead_time_list.append( int(my_file[-3:]) )
        print(members_list[-1],lead_time_list[-1])
    
    
# Run the merge
merge_gfs_ensemble(
    file_list , members_list , lead_time_list ,
    output_file,
    bbox=bbox,
    variables=variables,
    levels=levels
)
    
    

['0' '1' '2' '3' '4' '5' '6' '7' '8' '9' '10' '11' '12' '13' '14' '15'
 '16' '17' '18' '19' '20' '21' '22' '23' '24' '25' '26' '27' '28' '29'
 '30']
My member is  0
000 0
000 24
000 6
000 15
000 18
000 3
000 21
000 33
000 12
000 36
000 30
000 9
000 27
My member is  1
001 0
001 24
001 6
001 15
001 18
001 3
001 21
001 33
001 12
001 36
001 30
001 9
001 27
001 6
001 3
001 24
001 36
001 12
001 15
001 0
001 27
001 33
001 30
001 21
001 18
001 9
My member is  2
002 0
002 24
002 6
002 15
002 18
002 3
002 21
002 33
002 12
002 36
002 30
002 9
002 27
002 6
002 3
002 24
002 36
002 12
002 15
002 0
002 27
002 33
002 30
002 21
002 18
002 9
002 3
002 24
002 27
002 9
002 0
002 36
002 6
002 18
002 15
002 21
002 12
002 30
002 33
My member is  3
003 0
003 24
003 6
003 15
003 18
003 3
003 21
003 33
003 12
003 36
003 30
003 9
003 27
003 6
003 3
003 24
003 36
003 12
003 15
003 0
003 27
003 33
003 30
003 21
003 18
003 9
003 3
003 24
003 27
003 9
003 0
003 36
003 6
003 18
003 15
003 21
003 12
003 30
003 33
003 

skipping variable: paramId==260163 shortName='p260163'
Traceback (most recent call last):
  File "/home/jruiz/miniconda3/envs/gfs_env/lib/python3.10/site-packages/cfgrib/dataset.py", line 725, in build_dataset_components
    dict_merge(variables, coord_vars)
  File "/home/jruiz/miniconda3/envs/gfs_env/lib/python3.10/site-packages/cfgrib/dataset.py", line 641, in dict_merge
    raise DatasetBuildError(
cfgrib.dataset.DatasetBuildError: key present and new value is different: key='isobaricInhPa' value=Variable(dimensions=('isobaricInhPa',), data=array([1000.,  900.,  800.,  700.,  600.,  500.,  400.,  300.])) new_value=Variable(dimensions=('isobaricInhPa',), data=array([800., 700., 600., 500., 400., 300.]))
skipping variable: paramId==7001293 shortName='ICSEV'
Traceback (most recent call last):
  File "/home/jruiz/miniconda3/envs/gfs_env/lib/python3.10/site-packages/cfgrib/dataset.py", line 725, in build_dataset_components
    dict_merge(variables, coord_vars)
  File "/home/jruiz/minicon